# 02 · Extract to Volume

Worker notebook invoked by the **For Each** task — processes one API endpoint per iteration. Calls the Preqin Feed API, parses the CSV response, and writes the raw file to a UC Volume landing zone. Row count is captured at extract time and passed to downstream watermark/audit tasks via task values.

| Output | Description |
|---|---|
| `/Volumes/{catalog}/{schema}/preqin_landing/{api_domain}/{target_table}/` | Raw CSV landing zone in UC Volume |
| Task values | `status`, `domain`, `target_table`, `last_watermark_value`, `error_message`, `run_id`, `pipeline_start_time` |

**No direct Delta write** — Auto Loader pipeline (`PL_DEV_PREQIN_BRONZE`) picks up new files and loads them into Bronze Delta tables.

All parameters are supplied by the For Each task at runtime. Run cells sequentially.

In [0]:
# ---------------------------------------------------------------------------
# Widget definitions, imports, and endpoint data parsing.
# ---------------------------------------------------------------------------
import json
import time
import requests
from datetime import datetime, timedelta

max_retries = 2

BASE_URL = "https://feeds.preqin.com"

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
access_token = dbutils.widgets.get("access_token")

# Parse the endpoint data passed from the For Each task.
_raw = json.loads(dbutils.widgets.get("endpoint_data"))

endpoint_url     = _raw.get("endpoint_url")
api_domain       = _raw.get("api_domain")
target_table     = _raw.get("target_table")
api_version_used = _raw.get("api_version_used")
is_incremental   = bool(_raw.get("is_incremental"))
last_watermark_value = _raw.get("last_watermark_value")

url        = f"{BASE_URL}{endpoint_url}"
params     = {}

volume_dir = f"/Volumes/{catalog}/{schema}/preqin_landing/{api_domain}/{target_table}"

# Only create the volume if it doesn't already exist.
try:
    dbutils.fs.ls(f"/Volumes/{catalog}/{schema}/preqin_landing")
except Exception:
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.preqin_landing")

print(f"Processing: {api_domain}.{target_table} | Watermark: {last_watermark_value}")
print(f"Landing  : {volume_dir}")

In [0]:
def process_endpoint():
    """Process one API endpoint. Returns result dict."""

    result = {
        "status":               "",
        "error_message":        "",
        "last_watermark_value": "",
        "next_watermark_value": "",
        "watermark_value_used": "",
    }

    headers = {
        'Accept': 'text/csv',
        'Authorization': f'Bearer {access_token}',
        'Api-Version': api_version_used,
    }

    # Only pass date param when incremental AND watermark value exists.
    # Full refresh (is_incremental=False) or missing watermark → no date param.
    if is_incremental and last_watermark_value:
        params["date"] = last_watermark_value

    for attempt in range(max_retries + 1):
        try:
            response = requests.get(url, headers=headers, params=params, timeout=300)
            if response.status_code == 200:
                # Count newlines in raw bytes: header + at least one data row
                has_data = response.content.count(b'\n') > 1

                if has_data:
                    file_path = f"{volume_dir}/{target_table}_{datetime.now().strftime('%Y%m%d')}.csv"
                    dbutils.fs.put(file_path, response.content.decode("utf-8"), overwrite=True)
                    print(f"  Written → {file_path}")
                else:
                    print("  API returned 0 rows — no file written.")

                result["status"] = "SUCCESS"
                result["watermark_value_used"] = last_watermark_value
                result["next_watermark_value"] = (
                    datetime.now().strftime("%Y%m%d") if is_incremental else ""
                )
                break

            elif response.status_code == 401:
                result["error_message"] = "401 Token Expired"
                continue

            elif response.status_code == 403:
                result["status"]        = "SKIPPED_403"
                result["error_message"] = "403 Forbidden - Access denied"
                break

            elif response.status_code in (429, 500, 502, 503, 504):
                result["error_message"] = f"{response.status_code} Transient API Error"
                if attempt < max_retries:
                    time.sleep(2 ** attempt)
                    continue
                break

            else:
                result["status"]        = f"FAILED_{response.status_code}"
                result["error_message"] = f"{response.status_code} Unhandled response"
                break

        except requests.exceptions.Timeout:
            result["error_message"] = f"Timeout on attempt {attempt + 1}"
            if attempt < max_retries:
                time.sleep(2 ** attempt)
                continue
            result["status"] = "FAILED_TIMEOUT"
            break

        except Exception as e:
            result["status"]        = "FAILED_EXCEPTION"
            result["error_message"] = f"{type(e).__name__}: {str(e)[:500]}"
            break

    if not result["status"]:
        result["status"] = "FAILED_RETRIES_EXHAUSTED"

    print(result["status"])
    print(result["error_message"])

    return result

In [0]:
def __init__():
    required = {
        "endpoint_url": endpoint_url,
        "api_domain": api_domain,
        "target_table": target_table,
        "api_version_used": api_version_used,
    }
    missing = [k for k, v in required.items() if not v]
    if missing:
        return {
            "status":               "SKIPPED",
            "error_message":        f"Missing required fields: {', '.join(missing)}",
            "last_watermark_value": last_watermark_value or "",
        }
    return process_endpoint()

result = __init__()

In [0]:
# ---------------------------------------------------------------------------
# Used by: 03_preqin_api__watermark_update.sql, 04_insert_audit_log.sql
# ---------------------------------------------------------------------------
dbutils.jobs.taskValues.set(key="status",               value=result["status"])
dbutils.jobs.taskValues.set(key="domain",               value=api_domain or "")
dbutils.jobs.taskValues.set(key="endpoint_url",         value=endpoint_url or "")
dbutils.jobs.taskValues.set(key="target_table",         value=target_table or "")
dbutils.jobs.taskValues.set(key="last_watermark_value", value=result["next_watermark_value"]) # Watermark Update
dbutils.jobs.taskValues.set(key="watermark_value_used", value=result["watermark_value_used"]) # Audit Insert
dbutils.jobs.taskValues.set(key="error_message",        value=f"{result['status']}: {result['error_message']}" if result["error_message"] else "")

print("Task values set for downstream tasks.")

Task values set for downstream tasks.
